# Practical Exercise: Data Scraping (FluWatch)

**Task:** Scrape Table 2 from the FluWatch / Respiratory Virus Report page (week 01, ending January 5, 2019) into a pandas DataFrame, with one row for each province or territory.

Page: https://www.canada.ca/en/public-health/services/surveillance/respiratory-virus-detections-canada/2018-2019/respiratory-virus-detections-isolations-week-01-ending-january-5-2019.html

The exercise asks for one row per province or territory, so the plan is to pull the whole table first, then keep only the 13 provinces and territories.

In [31]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

# this is the URL we want to apply scraping
url = ("https://www.canada.ca/en/public-health/services/surveillance/"
       "respiratory-virus-detections-canada/2018-2019/"
       "respiratory-virus-detections-isolations-week-01-ending-january-5-2019.html")

## Pull the whole table into a DataFrame


In [25]:
tables = pd.read_html(url)

full = tables[1]

print(full.shape)
full.tail(10)

(42, 23)


,Reporting Laboratory,Flu Tested,A(H1N1)pdm09 Positive,A(H3) Positive,A(UnS) Positive,Total Flu A Positive,Total Flu B Positive,RSV Tested,RSV Positive,PIV Tested,...,PIV 4 Positive,Other PIV Positive,Adeno Tested,Adeno Positive,hMPV Tested,hMPV Positive,Entero/Rhino Tested,Entero/Rhino Positive,Coron Tested,Coron Positive
32,Province of Saskatchewan,8391,1280,60,749,2089,3,8391,160,10324,...,40,0,8391,106,8391,47,8391,1337,8391,30
33,Province of Alberta,18824,3132,73,1540,4745,29,15040,0,15040,...,154,0,15040,184,15040,189,15040,2693,15040,321
34,Prairies,32126,4604,136,2775,7515,35,28259,261,27618,...,204,0,25685,315,24645,245,25685,4340,24679,363
35,British Columbia,8731,1010,75,755,1840,10,8731,279,4342,...,66,0,4342,56,4342,14,3442,982,3442,54
36,Yukon,143,16,1,6,23,0,122,2,N.A.,...,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.
37,Northwest Territories,435,138,2,0,140,0,432,0,432,...,1,0,432,12,432,2,432,82,432,4
38,Nunavut,33,8,0,0,8,0,33,0,33,...,0,0,33,5,32,0,33,10,32,0
39,Territories,611,162,3,6,171,0,587,2,465,...,1,0,465,17,464,2,465,92,464,4
40,CANADA,110399,6609,395,10936,17941,172,103597,4753,60085,...,378,0,57982,979,56398,443,36098,6572,43923,855
41,"Specimens from YT, NT and NU are sent to refer...","Specimens from YT, NT and NU are sent to refer...","Specimens from YT, NT and NU are sent to refer...","Specimens from YT, NT and NU are sent to refer...","Specimens from YT, NT and NU are sent to refer...","Specimens from YT, NT and NU are sent to refer...","Specimens from YT, NT and NU are sent to refer...","Specimens from YT, NT and NU are sent to refer...","Specimens from YT, NT and NU are sent to refer...","Specimens from YT, NT and NU are sent to refer...",...,"Specimens from YT, NT and NU are sent to refer...","Specimens from YT, NT and NU are sent to refer...","Specimens from YT, NT and NU are sent to refer...","Specimens from YT, NT and NU are sent to refer...","Specimens from YT, NT and NU are sent to refer...","Specimens from YT, NT and NU are sent to refer...","Specimens from YT, NT and NU are sent to refer...","Specimens from YT, NT and NU are sent to refer...","Specimens from YT, NT and NU are sent to refer...","Specimens from YT, NT and NU are sent to refer..."


## Keep one row per province or territory


In [29]:
label_col = full.columns[0]

# Provinces/territories that appear under their own name
own_name = {
    "Newfoundland",
    "Prince Edward Island",
    "Nova Scotia",
    "New Brunswick",
    "Quebec",
    "Ontario",
    "Manitoba",
    "Saskatchewan",
    "Alberta",
    "British Columbia",
    "Yukon",
    "Northwest Territories",
    "Nunavut"
}


def tidy(label):
    """Collapse repeated spaces and drop the 'Province of ' prefix."""
    label = " ".join(str(label).split())
    if label.startswith("Province of "):
        label = label[len("Province of "):]
    return label

def is_province(label):
    name = " ".join(str(label).split())
    return name.startswith("Province of ") or name in own_name

provinces = full[full[label_col].apply(is_province)].copy()
provinces[label_col] = provinces[label_col].apply(tidy)
provinces = provinces.rename(columns={label_col: "Province/Territory"}).reset_index(drop=True)

print(provinces.shape)
provinces[["Province/Territory"]]

(13, 23)


,Province/Territory
0,Newfoundland
1,Prince Edward Island
2,Nova Scotia
3,New Brunswick
4,Québec
5,Ontario
6,Manitoba
7,Saskatchewan
8,Alberta
9,British Columbia


## Clean up the numbers

Every value came in as text. The counts should be numbers, and a few cells read "N.A." where a lab doesn't test for that virus. Converting with errors="coerce turns those into NaN and leaves the real counts as integers/floats.

In [30]:
count_cols = [c for c in provinces.columns if c != "Province/Territory"]
provinces[count_cols] = provinces[count_cols].apply(pd.to_numeric, errors="coerce")

provinces.info()
provinces

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Province/Territory     13 non-null     object 
 1   Flu Tested             13 non-null     int64  
 2   A(H1N1)pdm09 Positive  13 non-null     int64  
 3   A(H3) Positive         13 non-null     int64  
 4   A(UnS) Positive        13 non-null     int64  
 5   Total Flu A Positive   13 non-null     int64  
 6   Total Flu B Positive   13 non-null     int64  
 7   RSV Tested             13 non-null     int64  
 8   RSV Positive           13 non-null     int64  
 9   PIV Tested             12 non-null     float64
 10  PIV 1 Positive         12 non-null     float64
 11  PIV 2 Positive         12 non-null     float64
 12  PIV 3 Positive         12 non-null     float64
 13  PIV 4 Positive         12 non-null     float64
 14  Other PIV Positive     12 non-null     float64
 15  Adeno Te

,Province/Territory,Flu Tested,A(H1N1)pdm09 Positive,A(H3) Positive,A(UnS) Positive,Total Flu A Positive,Total Flu B Positive,RSV Tested,RSV Positive,PIV Tested,...,PIV 4 Positive,Other PIV Positive,Adeno Tested,Adeno Positive,hMPV Tested,hMPV Positive,Entero/Rhino Tested,Entero/Rhino Positive,Coron Tested,Coron Positive
0,Newfoundland,1299,1,0,113,114,1,1299,91,1299.0,...,0.0,0.0,1299.0,12.0,1299.0,8.0,1299.0,200.0,NaN,NaN
1,Prince Edward Island,307,38,0,0,38,0,305,5,53.0,...,3.0,0.0,48.0,5.0,48.0,0.0,48.0,21.0,48.0,0.0
2,Nova Scotia,864,0,0,52,52,1,869,45,322.0,...,1.0,0.0,322.0,0.0,322.0,3.0,322.0,53.0,322.0,1.0
3,New Brunswick,4271,42,1,715,758,2,4274,131,1185.0,...,29.0,0.0,1185.0,84.0,1185.0,7.0,1185.0,201.0,1185.0,6.0
4,Québec,38395,0,0,5974,5975,81,35755,2816,11201.0,...,36.0,0.0,11288.0,373.0,10786.0,63.0,NaN,NaN,10703.0,310.0
5,Ontario,23795,752,180,546,1478,42,23518,1123,13600.0,...,38.0,0.0,13348.0,117.0,13307.0,101.0,3652.0,683.0,3080.0,117.0
6,Manitoba,4911,192,3,486,681,3,4828,101,2254.0,...,10.0,0.0,2254.0,25.0,1214.0,9.0,2254.0,310.0,1248.0,12.0
7,Saskatchewan,8391,1280,60,749,2089,3,8391,160,10324.0,...,40.0,0.0,8391.0,106.0,8391.0,47.0,8391.0,1337.0,8391.0,30.0
8,Alberta,18824,3132,73,1540,4745,29,15040,0,15040.0,...,154.0,0.0,15040.0,184.0,15040.0,189.0,15040.0,2693.0,15040.0,321.0
9,British Columbia,8731,1010,75,755,1840,10,8731,279,4342.0,...,66.0,0.0,4342.0,56.0,4342.0,14.0,3442.0,982.0,3442.0,54.0


## Save the result

One tidy CSV with 13 rows, one per province or territory.

In [32]:
provinces.to_csv("fluwatch_week01_2019_by_province.csv", index=False)
print("Saved fluwatch_week01_2019_by_province.csv", provinces.shape)

Saved fluwatch_week01_2019_by_province.csv (13, 23)
